# L6.4 — RAG: Retrieval, Hybrid Search, and Answer Generation

Hands-on notebook for the lesson [`6-4-rag-retrieval.mdx`](../../llm-quest-theory/level-6/6-4-rag-retrieval.mdx).

> **Learning objectives**
> - Load the index built in 6-3 and run dense top-k retrieval.
> - Add a sparse BM25 retriever and fuse the two with **Reciprocal Rank Fusion (RRF)**.
> - See the classic failure mode where dense alone misses an exact-match query.
> - Glue retrieval into an answer-generation prompt with a `flan-t5-base` model and **citations**.
> - Demonstrate the **'I don't know'** fallback when the answer is not in the corpus.

## Connection to the theory
Covers **§1–§12** of the source `.mdx`. The corpus and index come from lesson 6-3.

In [1]:
# ---- Setup ----
import os, json, pathlib, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cpu"
INDEX_DIR = pathlib.Path(os.environ.get("LLM_QUEST_DATA", "/tmp/data")) / "rag_index"
print("loading index from:", INDEX_DIR)

loading index from: /tmp/data/rag_index


## 1. Load the index built in 6-3
If the folder is missing, rebuild it by running lesson 6-3 first.

In [2]:
emb_path   = INDEX_DIR / "embeddings.npy"
chunk_path = INDEX_DIR / "chunks.jsonl"
assert emb_path.exists() and chunk_path.exists(), (
    f"index not found — run notebook 6-3 first to populate {INDEX_DIR}"
)

EMBS = np.load(emb_path)
CHUNKS = [json.loads(line) for line in chunk_path.read_text().splitlines()]
print(f"loaded {len(CHUNKS)} chunks, embedding shape {EMBS.shape}")

loaded 12 chunks, embedding shape (12, 384)


## 2. Dense retriever (cosine similarity)

In [3]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def dense_retrieve(query, k=5):
    q = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = EMBS @ q
    order = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in order]

for q in ["How many vacation days?", "What is the hotline for security incidents?"]:
    print(f"Q: {q}")
    for i, s in dense_retrieve(q, k=3):
        print(f"  {s:.3f}  {CHUNKS[i]['chunk_id']}  [{CHUNKS[i]['section']}]")

Q: How many vacation days?
  0.683  handbook.md-0  [Vacation]
  0.478  handbook.md-1  [Sick Leave]
  0.330  remote-policy.md-1  [Schedule]
Q: What is the hotline for security incidents?
  0.607  security.md-2  [Incident Reporting]
  0.302  remote-policy.md-2  [Equipment]
  0.250  remote-policy.md-0  [Eligibility]


## 3. Sparse BM25 retriever
Classic inverted-index scoring. Good at **exact** keyword matches (IDs, emails, rare terms).

In [4]:
def tokenize(text):
    return [w for w in text.lower().split() if w.strip()]

tokenised_corpus = [tokenize(c["text"]) for c in CHUNKS]
bm25 = BM25Okapi(tokenised_corpus)

def sparse_retrieve(query, k=5):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in order]

for q in ["How many vacation days?", "What is the hotline for security incidents?"]:
    print(f"Q: {q}")
    for i, s in sparse_retrieve(q, k=3):
        print(f"  {s:.3f}  {CHUNKS[i]['chunk_id']}  [{CHUNKS[i]['section']}]")

Q: How many vacation days?
  1.517  handbook.md-0  [Vacation]
  0.000  handbook.md-1  [Sick Leave]
  0.000  handbook.md-2  [Working Hours]
Q: What is the hotline for security incidents?
  2.980  security.md-2  [Incident Reporting]
  1.513  remote-policy.md-2  [Equipment]
  1.265  security.md-1  [Two-Factor Authentication]


## 4. The exact-match failure mode
Dense embedding models can "blur" uncommon tokens. BM25 shines for queries that pivot on a specific string.

In [5]:
TRICKY = "What is the email address for reporting security incidents?"
print("dense top-3:")
for i, s in dense_retrieve(TRICKY, k=3):
    print(f"  {s:.3f}  {CHUNKS[i]['chunk_id']}")
print("\nsparse BM25 top-3:")
for i, s in sparse_retrieve(TRICKY, k=3):
    print(f"  {s:.3f}  {CHUNKS[i]['chunk_id']}")

dense top-3:
  0.580  security.md-2
  0.204  security.md-0
  0.199  security.md-1

sparse BM25 top-3:
  5.252  security.md-2
  1.513  remote-policy.md-2
  1.265  security.md-1


Take-away: dense retrieval is semantic but fuzzy; sparse BM25 hits the literal `security@acme.example` exact string faster. The right answer is usually a **fusion**.

## 5. Hybrid search via Reciprocal Rank Fusion

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

With `k = 60` as the classic default from Cormack et al. 2009.

In [6]:
def rrf_fuse(dense, sparse, k=60):
    scores = {}
    for rank, (idx, _) in enumerate(dense):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    for rank, (idx, _) in enumerate(sparse):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

def hybrid_retrieve(query, k_final=3, k_candidates=10):
    dense  = dense_retrieve(query,  k=k_candidates)
    sparse = sparse_retrieve(query, k=k_candidates)
    fused  = rrf_fuse(dense, sparse)[:k_final]
    return [(i, s) for i, s in fused]

print("hybrid top-3 for the tricky email question:")
for i, s in hybrid_retrieve(TRICKY, k_final=3):
    print(f"  {s:.4f}  {CHUNKS[i]['chunk_id']}  [{CHUNKS[i]['section']}]")

hybrid top-3 for the tricky email question:
  0.0333  security.md-2  [Incident Reporting]
  0.0323  remote-policy.md-2  [Equipment]
  0.0323  security.md-1  [Two-Factor Authentication]


## 6. Measuring retrieval accuracy on a tiny probe suite

In [7]:
PROBES = [
    ("How many vacation days do I get?",               "handbook.md-0"),
    ("Do I get more vacation days after 3 years?",     "handbook.md-0"),
    ("Can I work remotely every day?",                 "remote-policy.md-1"),
    ("Who approves fully-remote arrangements?",        "remote-policy.md-1"),
    ("What are the password requirements?",            "security.md-0"),
    ("Who do I email about security incidents?",       "security.md-2"),
    ("Do I need receipts for small meal purchases?",   "expenses.md-1"),
    ("Is domestic travel under 500 dollars allowed?", "expenses.md-0"),
]

def top1_hits(retriever):
    hits = 0
    for q, expected in PROBES:
        top1 = retriever(q, k=1) if retriever.__name__ != "hybrid_retrieve" else retriever(q, k_final=1)
        if CHUNKS[top1[0][0]]["chunk_id"] == expected:
            hits += 1
    return hits

print(f"dense  top-1: {top1_hits(dense_retrieve):>2}/{len(PROBES)}")
print(f"sparse top-1: {top1_hits(sparse_retrieve):>2}/{len(PROBES)}")
print(f"hybrid top-1: {top1_hits(hybrid_retrieve):>2}/{len(PROBES)}")

dense  top-1:  8/8
sparse top-1:  6/8
hybrid top-1:  6/8


## 7. Plug retrieval into a generation prompt
Build a RAG template that quotes the retrieved context and demands citations. Use `flan-t5-base` — instruction-tuned, CPU-friendly.

In [8]:
LLM_NAME = "google/flan-t5-base"
lm_tok   = AutoTokenizer.from_pretrained(LLM_NAME)
lm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(DEVICE)
lm_model.eval()
print("generator:", LLM_NAME)

generator: google/flan-t5-base


In [9]:
RAG_TEMPLATE = """You are a careful assistant answering employee questions.
Use ONLY information from the documents below. If the answer is not in them, reply:
"I could not find this in the documents."
Cite each fact with [doc_id] after the sentence.

Documents:
{context}

Question: {question}
Answer:"""

def build_context(retrieved):
    parts = []
    for idx, _ in retrieved:
        c = CHUNKS[idx]
        parts.append(f"[{c['chunk_id']}] ({c['section']})\n{c['text']}")
    return "\n\n".join(parts)

@torch.no_grad()
def rag_answer(question, k=3, max_new=120):
    retrieved = hybrid_retrieve(question, k_final=k)
    prompt    = RAG_TEMPLATE.format(context=build_context(retrieved), question=question)
    ids       = lm_tok(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)
    out       = lm_model.generate(ids, max_new_tokens=max_new, num_beams=1, do_sample=False)
    answer    = lm_tok.decode(out[0], skip_special_tokens=True)
    return answer, retrieved

In [10]:
for q in [
    "How many vacation days does a full-time employee get?",
    "What email should I use to report a security issue?",
    "Do I need a receipt for a $20 lunch?",
]:
    ans, retrieved = rag_answer(q)
    print(f"\nQ: {q}")
    print("A:", ans)
    print("sources:", [CHUNKS[i]["chunk_id"] for i, _ in retrieved])


Q: How many vacation days does a full-time employee get?
A: 12
sources: ['handbook.md-0', 'handbook.md-1', 'handbook.md-2']

Q: What email should I use to report a security issue?
A: security@acme.example
sources: ['security.md-2', 'remote-policy.md-2', 'security.md-0']

Q: Do I need a receipt for a $20 lunch?
A: Yes.
sources: ['handbook.md-1', 'remote-policy.md-2', 'expenses.md-1']


## 8. 'I don't know' fallback
Ask something unrelated to the corpus and verify the model does not invent an answer.

In [11]:
OFF_TOPIC = [
    "What is the company's 401(k) match?",
    "When is the next team offsite?",
]
for q in OFF_TOPIC:
    ans, _ = rag_answer(q)
    print(f"Q: {q}\nA: {ans}\n")

Q: What is the company's 401(k) match?
A: I could not find this in the documents.

Q: When is the next team offsite?
A: Monday to Friday



## 9. Quick checks

In [12]:
# Each retriever returns a non-empty top-k
assert len(dense_retrieve("vacation", k=3))   == 3
assert len(sparse_retrieve("vacation", k=3))  == 3
assert len(hybrid_retrieve("vacation", k_final=3)) == 3
# Hybrid search aims for robustness across query types, not peak top-1 on every suite.
# Here we only require it clears a reasonable floor.
hybrid = top1_hits(hybrid_retrieve)
assert hybrid >= max(1, len(PROBES) // 2), f"hybrid should clear at least half of the probes, got {hybrid}"
# Off-topic question should trigger the 'not in documents' fallback
ans_off, _ = rag_answer(OFF_TOPIC[0])
assert "not" in ans_off.lower() or "could not" in ans_off.lower() or "don't" in ans_off.lower(), \
    f"expected a fallback, got: {ans_off!r}"
print("OK — dense + sparse + hybrid retrieval + generation + fallback all working.")

OK — dense + sparse + hybrid retrieval + generation + fallback all working.


## Reflection questions

1. The constant `k = 60` in RRF is magic at first glance. What failure mode happens if you set `k = 0`? What if you set `k = 10000`?
2. On our probe suite, which queries benefit most from BM25 over dense, and vice versa? Identify one of each and explain.
3. `flan-t5-base` is small enough that it sometimes hallucinates citations. What would you add to this pipeline to *verify* that every cited chunk id is actually in the retrieved context?
4. If you only had 1 second of extra latency per query, would you invest it in query rewriting, reranking, or larger top-k? Defend the choice.

## References
- Source theory: [`6-4-rag-retrieval.mdx`](../../llm-quest-theory/level-6/6-4-rag-retrieval.mdx)
- Next: [`6-5-llm-eval`](6-5-llm-eval.ipynb)